In [3]:
pip install requests langgraph


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
#Configuration

SARVAM_API_KEY = "sk_jt9u2jn9_ZSb5E7Jt2pOYdldMXAX1nKUL"
MODEL_NAME = "sarvam-105b"


In [ ]:
# 1. Define the Shared Memory (The 'State')

class SwarmState(TypedDict):
    business_idea: str
    market_research: str
    brand_identity : str

In [ ]:
# 2. HELPER: The Sarvam Brain Function

def ask_sarvam(prompt):
    url = "https://api.sarvam.ai/v1/chat/completions"
    headers = {'api-subscription-key": sk_jt9u2jn9_ZSb5E7Jt2pOYdldMXAX1nKUL}
    payload = {
        model
    

In [ ]:
import requests
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# --- 1. SHARED MEMORY ---
class SwarmState(TypedDict):
    business_idea: str
    market_research: str
    brand_identity: str

# --- 2. THE BRAIN (Now using the correct 2026 headers) ---
def call_sarvam_api(prompt_text):
    # HARDCODE YOUR KEY HERE JUST TO TEST
    MY_KEY = "sk_jt9u2jn9_ZSb5E7Jt2pOYdldMXAX1nKUL" 
    
    url = "https://api.sarvam.ai/v1/chat/completions"
    headers = {
        "api-subscription-key": MY_KEY,
        "Content-Type": "application/json"
    }
    payload = {
        "model": "sarvam-105b",
        "messages": [{"role": "user", "content": prompt_text}]
    }
    
    response = requests.post(url, json=payload, headers=headers)
    
    # Troubleshooting: if it fails, this will tell us why
    if response.status_code != 200:
        return f"Error: {response.status_code} - {response.text}"
        
    return response.json()['choices'][0]['message']['content']

# --- 3. THE AGENTS ---

def researcher(state: SwarmState):
    print("--- 🔍 Researcher is analyzing... ---")
    query = f"Provide a brief market analysis for: {state['business_idea']}"
    # We call the function directly
    data = call_sarvam_api(query)
    return {"market_research": data}

def strategist(state: SwarmState):
    print("--- 🎨 Strategist is designing... ---")
    research = state["market_research"]
    query = f"Based on this research: {research}, give me a Brand Name and Slogan."
    data = call_sarvam_api(query)
    return {"brand_identity": data}

# --- 4. THE FLOW ---

workflow = StateGraph(SwarmState)

workflow.add_node("researcher", researcher)
workflow.add_node("strategist", strategist)

workflow.add_edge(START, "researcher")
workflow.add_edge("researcher", "strategist")
workflow.add_edge("strategist", END)

# Compile
app = workflow.compile()

# Run
test_input = {"business_idea": "Budget travel agency for students in Hyderabad"}
output = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---")
print(output["brand_identity"])

--- 🔍 Researcher is analyzing... ---
--- 🎨 Strategist is designing... ---
